In [3]:
# Configuración inicial
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
import os
import time

# --- IMPORTACIONES ---
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler


stream_handler = StreamingStdOutCallbackHandler()

# Configurar el modelo
llm = ChatOpenAI(
    base_url=os.getenv("OPENAI_BASE_URL"),
    api_key=os.getenv("GITHUB_TOKEN"),
    model="gpt-4.1",
    streaming=True,
    callbacks=[stream_handler],
    request_timeout=600,
    temperature=0.1  # Baja temperatura para razonamiento más consistente
)


print("✓ Modelo configurado para Chain-of-Thought")
print("✓ Temperature baja para razonamiento consistente")

✓ Modelo configurado para Chain-of-Thought
✓ Temperature baja para razonamiento consistente


In [12]:
# Few-shot CoT: ejemplos con razonamiento paso a paso
def few_shot_cot():
    print("=== FEW-SHOT CHAIN-OF-THOUGHT ===")
    
    # Nuevo problema para resolver
    nuevo_problema = "Cual es mi nombre y donde puedo ir a visitar?"
    
    # Prompt con ejemplos de razonamiento
    prompt = f"""Resuelve la problematica del turista mostrando el razonamiento paso a paso:
    
Problema: El turista de nombre Matias quiere ir a visitar al rededor de Puerto Montt, ¿qué lugares puede visitar?
Razonamiento:
1. Puerto Montt es una ciudad ubicada en el sur de Chile, conocida por su belleza natural y su cercanía a diversos atractivos turísticos.
2. Algunos de los lugares más populares para visitar alrededor de Puerto Montt incluyen:
   - Isla Tenglo: una pequeña isla frente a Puerto Montt, ideal para paseos y disfrutar de vistas panorámicas.
   - Parque Nacional Alerce Andino: un parque nacional que alberga bosques de alerces milenarios, senderos para caminatas y una rica biodiversidad.
   - Chiloé: una isla cercana a Puerto Montt, famosa por su arquitectura tradicional, su cultura única y sus hermosos paisajes.
   - Lago Llanquihue: un lago grande y pintoresco cerca de Puerto Montt, rodeado de montañas y con varias actividades acuáticas disponibles.
3. Además, el turista puede disfrutar de la gastronomía local, que incluye mariscos frescos y platos típicos de la región.
Respuesta: El turista puede visitar Isla Tenglo, Parque Nacional Alerce Andino, Chiloé y Lago Llanquihue, además de disfrutar de la gastronomía local.
    
Problema: El turista de nombre Matias pregunta donde puede ir a tomarse fotos en Puerto Montt o en las localidades cercanas
Razonamiento:
1. Puede visitar el Mirador de Puerto Montt, que ofrece vistas panorámicas de la ciudad y el mar.
2. El Muelle Angelmó es otro lugar popular para tomar fotos, con su ambiente colorido y su mercado de mariscos.
3. La Isla Tenglo también es un excelente lugar para capturar fotos de la naturaleza y las vistas al mar.
4. En las localidades cercanas, el Parque Nacional Alerce Andino ofrece paisajes impresionantes y bosques milenarios que son perfectos para la fotografía.
Respuesta: El turista puede visitar el Mirador de Puerto Montt, el Muelle Angelmó, la Isla Tenglo y el Parque Nacional Alerce Andino para tomarse fotos.
    
Problema: {nuevo_problema}
Razonamiento:"""
    
    start_time = time.time()
    response_text = ""
    try:
        for chunk in llm.invoke([HumanMessage(content=prompt)]):
            print(chunk.content, end="", flush=True)
            response_text += chunk.content
            time.sleep(0.03)  # Simular pausa para efecto visual

        print("\nPROBLEMA A RESOLVER:")
        print(nuevo_problema)
        print("\nSOLUCIÓN CON RAZONAMIENTO:")
        print(response_text)

        # Verificar si siguió el patrón
        razonamiento = response_text
        tiene_pasos_numerados = bool(re.search(r'\d+\.', razonamiento))
        tiene_calculos = any(op in razonamiento for op in ['=', '+', '-', '*', '/', '€'])
        tiene_respuesta_final = 'respuesta' in razonamiento.lower()

        end_time = time.time()
        print(f"\\n\\n[Streaming completado en {end_time - start_time:.2f} segundos]")
        
        print("\n=== ANÁLISIS DEL PATRÓN ===")
        print(f"✓ Pasos numerados: {'Sí' if tiene_pasos_numerados else 'No'}")
        print(f"✓ Respuesta final clara: {'Sí' if tiene_respuesta_final else 'No'}")
        
    except Exception as e:
        print(f"Error: {e}")

# Ejecutar few-shot CoT
import re       
few_shot_cot()

=== FEW-SHOT CHAIN-OF-THOUGHT ===
Razonamiento:

1. Según la información anterior, tu nombre es Matias.
2. Quieres saber dónde puedes ir a visitar.
3. Ya se ha mencionado que Puerto Montt y sus alrededores tienen varios lugares turísticos interesantes.
4. Entre los lugares recomendados están:
   - Isla Tenglo: ideal para paseos y vistas panorámicas.
   - Parque Nacional Alerce Andino: para disfrutar de la naturaleza y hacer caminatas.
   - Chiloé: una isla cercana con cultura y paisajes únicos.
   - Lago Llanquihue: para actividades acuáticas y vistas a los volcanes.
   - Mirador de Puerto Montt: para fotos panorámicas de la ciudad.
   - Muelle Angelmó: para conocer el mercado y la gastronomía local.
5. Además, puedes disfrutar de la gastronomía típica de la zona, especialmente los mariscos.

Respuesta:  
Tu nombre es Matias y puedes ir a visitar Isla Tenglo, Parque Nacional Alerce Andino, Chiloé, Lago Llanquihue, el Mirador de Puerto Montt y el Muelle Angelmó. Además, puedes disfrutar